# Record trajectory

Record a trajectory from the robot or teleoperation and optionally save it. This mirrors `examples/scripts/record_trajectory.py`.

Ensure the robot is clear before starting to record.


In [ ]:
import sys
import time

from r2_labs import client, rpc_api

host = "akhilraju-home.local"
server_address = f"tcp://{host}:{rpc_api.DEFAULT_PORT}"
query_address = f"tcp://{host}:{rpc_api.DEFAULT_QUERY_PORT}"
training_address = f"tcp://{host}:{rpc_api.DEFAULT_MODEL_TRAINER_PORT}"

robot = client.Robot(
    server_address,
    query_server_address=query_address,
    training_server_address=training_address,
)


In [ ]:
def get_trajectory_type(value: str) -> rpc_api.TrajectoryType:
  match value:
    case "joint_absolute":
      return rpc_api.TrajectoryType.JOINT_ABSOLUTE
    case "joint_relative":
      return rpc_api.TrajectoryType.JOINT_RELATIVE
    case "wrist_cartesian_relative":
      return rpc_api.TrajectoryType.WRIST_CARTESIAN_RELATIVE
  raise ValueError(f"Unknown trajectory type: {value}")


def get_trajectory_source(value: str) -> rpc_api.TrajectorySource:
  match value:
    case "robot":
      return rpc_api.TrajectorySource.ROBOT
    case "teleop":
      return rpc_api.TrajectorySource.TELEOP
  raise ValueError(f"Unknown trajectory source: {value}")


In [ ]:
# Options for the recording. Modify these as needed before running the recording
# cell.
name = "akhil_test"  # required when save is True
description = ""
# joint_absolute | joint_relative | wrist_cartesian_relative
trajectory_type = "wrist_cartesian_relative"
trajectory_source = "teleop"  # robot | teleop
timeout_seconds = 30.0  # 0 for no timeout
save = True
overwrite = True
hold_until_start = True

## Move robot before recording

In [ ]:
# Optional step to move the robot to a starting position before recording.
# Left blank here, but feel free to modify this to fit your needs.

# e.g.
# motion_future = None

# try:
#   # Move to a predefined "home" position.
#   robot.exec_mode.set_execution_mode(rpc_api.ExecutionMode.READY)
#   motion_future = robot.arm.trajectory_motion(
#     trajectory_name="view_printer_front_from_above",
#     period_seconds=None,
#     motion_type=rpc_api.TrajectoryMotionType.GO_TO_END,
#     static_gripper=False,
#   )
#   motion_future.result()
#   # Then move to a visual pose.
#   robot.exec_mode.set_execution_mode(rpc_api.ExecutionMode.READY)
#   motion_future = robot.arm.visual_pose_motion(
#     visual_pose_name="view_front_tag_from_above",
#     period_seconds=5.0,
#   )
#   motion_future.result()
#   # Then align the leader with the follower (assuming "teleop")
#   robot.exec_mode.set_execution_mode(rpc_api.ExecutionMode.READY)
#   motion_future = robot.behaviour.align_leader_with_follower(
#     timeout_seconds=2.0,
#     threshold=0.05,
#   )
#   motion_future.result()
# except Exception as e:
#   print(f"Error during optional setup motions: {e}")
#   if motion_future is not None:
#     motion_future.cancel()

## Prepare recording

Call this before you record a new trajectory.

In [ ]:
trajectory_type_enum = get_trajectory_type(trajectory_type)
trajectory_source_enum = get_trajectory_source(trajectory_source)
timeout = timeout_seconds if timeout_seconds > 0 else None

if save and not name:
  raise ValueError("--name is required when save is True")

print("Preparing to record trajectory...")
print(f"  Type: {trajectory_type}")
print(f"  Source: {trajectory_source}")
print(f"  Timeout: {timeout}s" if timeout else "  Timeout: None")
print(f"  Hold until start: {hold_until_start}")

prepare_response = robot.recording.prepare(
    trajectory_type=trajectory_type_enum,
    trajectory_source=trajectory_source_enum,
    timeout_seconds=timeout,
    hold_until_start=hold_until_start,
)

if prepare_response.error:
  raise RuntimeError(f"Error preparing recording: {prepare_response.error}")


## Record trajectory

In [ ]:

start_response = robot.recording.start()
if start_response.error:
  raise RuntimeError(f"Error starting recording: {start_response.error}")

print("Recording... stop/interrupt the cell to end (Jupyter stop or Kernel > Interrupt).")
start_time = time.time()

try:
  while True:
    state = robot.recording.get_state()
    if not state.is_recording:
      if state.timed_out:
        print("\nRecording timed out")
      break

    elapsed = time.time() - start_time
    print(f"\r  Samples: {state.sample_count}  Elapsed: {elapsed:.1f}s", end="", flush=True)
    time.sleep(0.2)
except KeyboardInterrupt:
  print("\nStopping recording...")
finally:
  stop_response = robot.recording.stop()
  print("Recording stopped.")


You can set the robot down after stopping the recording. You don't have to end recorded trajectories in the home position.

## Save trajectory

In [ ]:

if stop_response.error:
  raise RuntimeError(f"Error stopping recording: {stop_response.error}")

trajectory = stop_response.trajectory
if trajectory is None:
  raise RuntimeError("No trajectory recorded")

print("\nRecorded trajectory:")
print(f"  Duration: {trajectory.period_seconds:.2f}s")
print(f"  Samples: {trajectory.trajectory_data.shape[0]}")
print(f"  Type: {trajectory.trajectory_type.name}")
print(f"  Source: {trajectory.trajectory_source.name}")

if save:
  if not name:
    raise ValueError("Name is required to save trajectory")
  trajectory.name = name
  trajectory.description = description

  print(f"\nSaving trajectory as '{name}'...")
  add_response = robot.trajectory_library.add_entry(
      trajectory=trajectory,
      allow_overwrite=overwrite,
  )

  if add_response.success:
    print("Trajectory saved successfully")
  else:
    raise RuntimeError("Failed to save trajectory (name may already exist)")
else:
  print("\nTrajectory not saved (save is False)")

print("Done")

## Replay trajectory

If you moved the robot before recording the trajectory, and you recorded a relative trajectory, be sure to move the robot back to that position before replaying again.

In [ ]:
if trajectory is None:
  raise RuntimeError("No trajectory recorded, cannot execute")

trajectory_name = name
period_seconds = trajectory.period_seconds
motion_type = rpc_api.TrajectoryMotionType.FULL
static_gripper = False

motion_future = None

try:
  robot.exec_mode.set_execution_mode(rpc_api.ExecutionMode.READY)
  motion_future = robot.arm.trajectory_motion(
      trajectory_name=trajectory_name,
      period_seconds=period_seconds,
      motion_type=motion_type,
      static_gripper=static_gripper,
  )
  motion_future.result()
except Exception as e:
  print(f"Error executing trajectory: {e}")
finally:
  if motion_future and not motion_future.done():
    motion_future.cancel()